# Dependencies

In [1]:
import pandas as pd

# Combine Transport data with demographic data

In [5]:
demo_info = pd.read_csv('Data/demographics.csv')
transport_data = pd.read_csv('Data/transport_data.csv')

def join_district_info(df, demo_info):
    df = df.merge(demo_info, how='left', left_on='DISTRICT', right_on='CONSTITUENCY')
    df = df.merge(demo_info, how='left', left_on='NEXT_DISTRICT', right_on='CONSTITUENCY', suffixes=('', '_NEXT'))
    return df

distance_upper80 = transport_data["DISTANCE"].quantile(0.8)
distance_lower10 = transport_data["DISTANCE"].quantile(0.1)
price_upper90 = transport_data["PRICE"].quantile(0.9)
price_lower1= transport_data["PRICE"].quantile(0.01)
transport_data = transport_data[(transport_data['DISTANCE'] >= distance_lower10) & 
                                 (transport_data['DISTANCE'] <= distance_upper80) &
                                 (transport_data['PRICE'] >= price_lower1) &
                                 (transport_data['PRICE'] <= price_upper90)]

transport_data = join_district_info(transport_data, demo_info)

transport_data = transport_data.drop(columns=['CONSTITUENCY', 'CONSTITUENCY_NEXT'])
transport_data = transport_data.drop(
       columns=['working population ratio ()', 
                'employed population (by 18 districts)',
                'building number (by 18 districts)', 
                'empolyed population ratio ()',
                'working population ratio ()_NEXT', 
                'employed population (by 18 districts)_NEXT',
                'building number (by 18 districts)_NEXT', 
                'empolyed population ratio ()_NEXT'])


# Rename DISTRICT and NEXT_DISTRICT column
transport_data = transport_data.rename(
       columns={
           'DISTRICT': 'DISTRICT',
           'NEXT_DISTRICT': 'NEXT DISTRICT', 
           'living population': 'LIVING POPULATION',
           'working population': 'WORKING POPULATION',
           'building number': 'BUILDING NUMBER',
           'estimated employed population': 'ESTIMATED EMPLOYED POPULATION',
           'median monthly income (18 district)': 'MEDIAN MONTHLY INCOME',
           'STOP_COUNT_NEXT': 'NEXT STOP_COUNT', 
           'STOP_COUNT_UNIQUE_NEXT': 'NEXT STOP_COUNT_UNIQUE',
           'AREA_NEXT': 'NEXT AREA',
           'DENSITY_NEXT': 'NEXT DENSITY',
           'DENSITY_UNIQUE_NEXT': 'NEXT DENSITY_UNIQUE',
           'DENSITY*10000_NEXT': 'NEXT DENSITY*10000',
           'DENSITY_UNIQUE*100000_NEXT': 'NEXT DENSITY_UNIQUE*100000', 
           'living population_NEXT': 'NEXT LIVING POPULATION',
           'working population_NEXT': 'NEXT WORKING POPULATION',
           'building number_NEXT': 'NEXT BUILDING NUMBER',
           'estimated employed population_NEXT': 'NEXT ESTIMATED EMPLOYED POPULATION',
           'median monthly income (18 district)_NEXT': 'NEXT MEDIAN MONTHLY INCOME'
       })

# Remove DISTRICT == 'shenzhen'
transport_data = transport_data[transport_data['DISTRICT'] != 'Shenzhen']
transport_data = transport_data[transport_data['NEXT DISTRICT'] != 'Shenzhen']

# Add 'PASS HARBOUR' column based on the specified conditions
SEA_DISTRICT = ['Southern District Southeast', 'Western', 'Hong Wan', 'Central', 'Islands', 'Wan Chai', 'Southern District Northwest', 'Chai Wan', 'Tai Pak']
transport_data['PASS HARBOUR'] = transport_data.apply(
    lambda x: 1 if (x['DISTRICT'] in SEA_DISTRICT and x['NEXT DISTRICT'] not in SEA_DISTRICT) or 
                  (x['DISTRICT'] not in SEA_DISTRICT and x['NEXT DISTRICT'] in SEA_DISTRICT) else 0, 
    axis=1
)

transport_data.to_csv('Data/final_transport_data.csv', index=False)